# Holdout-safe time-series EDA

This notebook examines the validated state-quarter panel before baseline implementation. Structural coverage may be inspected through `2026Q2`, but production magnitudes are available only from the modeling start through the latest development target, `2022Q1`. It does not impute or remove observations, select a transformation or model, or calculate holdout performance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Set the same private Drive root used by notebook 01. This notebook requires the validated checkpoints and passed run manifest produced there.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ds_portfolio/project_02_coal_production_forecasting')
DATA_ROOT = DRIVE_PROJECT_ROOT / 'data'
RUNS_ROOT = DRIVE_PROJECT_ROOT / 'runs'

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/ahmaddshbg-blip/regional-coal-production-forecasting.git'
REPO_DIR = Path('/content/regional-coal-production-forecasting')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-lock.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--no-deps'],
    check=True,
)
source_root = str(REPO_DIR / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

revision = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
print('Code revision:', revision)

In [ ]:
import os

os.chdir(REPO_DIR)
os.environ['PROJECT_DATA_ROOT'] = str(DATA_ROOT)
os.environ['PROJECT_RUNS_ROOT'] = str(RUNS_ROOT)

subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=REPO_DIR, check=True
)

## Validated checkpoint and leakage boundary

The latest development target is derived from the final validation origin plus the maximum forecast horizon. Keeping this derivation in code prevents an accidental drift between EDA and the frozen forecasting contract.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter

from coal_forecasting import load_latest_validated_manifest
from coal_forecasting.config import load_config
from coal_forecasting.eda import (
    add_quarters,
    build_coverage_grid,
    development_values,
    eligibility_transitions,
    origin_eligibility,
    period_label,
    period_start,
    quarter_starts,
    quarterly_concentration,
    seasonal_quarter_share,
    state_coverage_summary,
    state_target_summary,
    temporal_change_diagnostics,
)
from coal_forecasting.paths import resolve_pipeline_paths

CONFIG_PATH = REPO_DIR / 'configs' / 'project.json'
config = load_config(CONFIG_PATH)
manifest = load_latest_validated_manifest(CONFIG_PATH, root=REPO_DIR)
paths = resolve_pipeline_paths(config, REPO_DIR)
evaluation = config['evaluation']

DEVELOPMENT_TARGET_END = add_quarters(
    evaluation['validation_origins']['end'],
    max(evaluation['forecast_horizons']),
)
DEVELOPMENT_CUTOFF = period_start(DEVELOPMENT_TARGET_END)
assert DEVELOPMENT_TARGET_END == '2022Q1'
assert manifest['status'] == 'passed'
print('Validated run:', manifest['run_id'])
print('Magnitude window:', evaluation['modeling_start'], 'through', DEVELOPMENT_TARGET_END)
print('Structural window:', config['snapshot']['coverage_start'], 'through', config['snapshot']['coverage_end'])

In [ ]:
# Later magnitudes are replaced in DuckDB before any target frame enters pandas.
with duckdb.connect() as connection:
    guarded_panel = connection.execute(
        '''
        SELECT
            state_code, cal_year, cal_quarter, period, quarter_start_date,
            CASE
                WHEN quarter_start_date <= CAST(? AS DATE)
                    THEN coal_production_short_tons
                WHEN coal_production_short_tons IS NULL THEN NULL
                ELSE 1.0
            END AS coal_production_short_tons
        FROM read_parquet(?)
        ORDER BY state_code, quarter_start_date
        ''',
        [DEVELOPMENT_CUTOFF.date(), str(paths['state_quarter'])],
    ).fetchdf()

post_cutoff_guard = guarded_panel.loc[
    guarded_panel['quarter_start_date'].gt(DEVELOPMENT_CUTOFF),
    'coal_production_short_tons',
]
assert post_cutoff_guard.dropna().eq(1.0).all()
assert len(guarded_panel) == config['quality_expectations']['state_quarter_rows']
print('Guarded state-quarter rows:', len(guarded_panel))

## Structural coverage

A complete state-quarter grid separates leading absence, gaps inside a state's reporting span, trailing absence, reported zero, and reported positive production. Quarters after `2022Q1` show presence only; their production magnitudes remain masked.

In [ ]:
coverage = build_coverage_grid(
    guarded_panel,
    start_period=config['snapshot']['coverage_start'],
    end_period=config['snapshot']['coverage_end'],
    value_end_period=DEVELOPMENT_TARGET_END,
)
coverage_summary = state_coverage_summary(coverage)
assert coverage.loc[
    coverage['quarter_start_date'].gt(DEVELOPMENT_CUTOFF), 'eda_target'
].isna().all()

status_counts = (
    coverage['coverage_status'].value_counts().rename_axis('coverage_status')
    .rename('quarters').to_frame()
)
display(status_counts)
display(coverage_summary)

status_order = [
    'leading_absence', 'internal_absence', 'trailing_absence',
    'observed_null', 'observed_zero', 'observed_positive', 'observed_masked',
]
status_colors = [
    '#e5e7eb', '#f59e0b', '#9ca3af', '#dc2626',
    '#38bdf8', '#15803d', '#1f2937',
]
status_labels = [
    'leading absence', 'internal absence', 'trailing absence',
    'observed null', 'reported zero', 'reported positive',
    'row present, magnitude masked',
]
status_codes = {status: index for index, status in enumerate(status_order)}
coverage_matrix = (
    coverage.assign(status_code=coverage['coverage_status'].map(status_codes))
    .pivot(index='state_code', columns='quarter_start_date', values='status_code')
)
fig, ax = plt.subplots(figsize=(15, 7))
ax.imshow(
    coverage_matrix.to_numpy(), aspect='auto', interpolation='nearest',
    cmap=plt.matplotlib.colors.ListedColormap(status_colors),
    vmin=-0.5, vmax=len(status_order) - 0.5,
)
tick_positions = list(range(0, len(coverage_matrix.columns), 8))
if len(coverage_matrix.columns) - 1 - tick_positions[-1] >= 4:
    tick_positions.append(len(coverage_matrix.columns) - 1)
ax.set_xticks(tick_positions)
ax.set_xticklabels([period_label(coverage_matrix.columns[i]) for i in tick_positions], rotation=45, ha='right')
ax.set_yticks(range(len(coverage_matrix.index)))
ax.set_yticklabels(coverage_matrix.index)
ax.set(title='State-quarter structural coverage', xlabel='', ylabel='State')
ax.legend(
    handles=[Patch(color=color, label=label) for color, label in zip(status_colors, status_labels)],
    loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False,
)
fig.tight_layout()

## Development-window target behavior

All figures and summaries below use production values no later than `2022Q1`. Reported zeros stay as observations. No value is imputed, clipped, winsorized, or removed.

In [ ]:
development = development_values(
    guarded_panel,
    start_period=evaluation['modeling_start'],
    end_period=DEVELOPMENT_TARGET_END,
)
state_summary = state_target_summary(
    development, value_end_period=DEVELOPMENT_TARGET_END
)
assert development['quarter_start_date'].max() == DEVELOPMENT_CUTOFF

development_overview = pd.Series(
    {
        'state-quarter rows': len(development),
        'states': development['state_code'].nunique(),
        'first period': development['period'].min(),
        'last period': development['period'].max(),
        'null targets': int(development['coal_production_short_tons'].isna().sum()),
        'reported zero targets': int(development['coal_production_short_tons'].eq(0).sum()),
    },
    name='value',
)
display(development_overview.to_frame())
state_summary_view = state_summary.assign(
    total_million_tons=state_summary['total_production'] / 1_000_000,
    mean_million_tons=state_summary['mean_production'] / 1_000_000,
    maximum_million_tons=state_summary['maximum_production'] / 1_000_000,
)[
    ['state_code', 'first_period', 'last_period', 'observed_periods',
     'zero_periods', 'zero_share', 'total_million_tons',
     'mean_million_tons', 'maximum_million_tons']
].round(3)
display(state_summary_view)

In [ ]:
national = (
    development.groupby('quarter_start_date', as_index=False)['coal_production_short_tons']
    .sum(min_count=1)
)
leaders = state_summary.head(8)['state_code'].tolist()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(national['quarter_start_date'], national['coal_production_short_tons'] / 1_000_000, color='#111827', linewidth=2)
ax.axvline(DEVELOPMENT_CUTOFF, color='#dc2626', linestyle='--', linewidth=1)
ax.set(title=f'National aggregate within the development window ({evaluation["modeling_start"]}-{DEVELOPMENT_TARGET_END})', xlabel='', ylabel='Million short tons')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
for ax, state in zip(axes.flat, leaders):
    series = development.loc[development['state_code'].eq(state)]
    ax.plot(series['quarter_start_date'], series['coal_production_short_tons'] / 1_000_000, linewidth=1.6)
    ax.set_title(state)
    ax.set_ylabel('Million tons')
    ax.grid(axis='y', alpha=0.2)
fig.suptitle('Largest states by development-window production', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])

In [ ]:
seasonality = seasonal_quarter_share(
    development, value_end_period=DEVELOPMENT_TARGET_END
)
concentration = quarterly_concentration(
    development, value_end_period=DEVELOPMENT_TARGET_END
)
display(seasonality.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(seasonality['cal_quarter'], seasonality['median'], color=['#2563eb', '#059669', '#d97706', '#7c3aed'])
axes[0].errorbar(
    seasonality['cal_quarter'], seasonality['median'],
    yerr=np.vstack([seasonality['median'] - seasonality['q25'], seasonality['q75'] - seasonality['median']]),
    fmt='none', ecolor='#111827', capsize=4, linewidth=1,
)
axes[0].set(xticks=[1, 2, 3, 4], xlabel='Calendar quarter', ylabel='Median share of annual production', title='Seasonal share across complete state-years')
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].grid(axis='y', alpha=0.2)

axes[1].plot(concentration['quarter_start_date'], concentration['top5_share'], label='Top-five state share', color='#0f766e')
axes[1].plot(concentration['quarter_start_date'], concentration['hhi'], label='HHI', color='#b45309')
axes[1].set(title='Quarterly regional concentration', xlabel='', ylabel='Share / index', ylim=(0, 1))
axes[1].legend(frameon=False)
axes[1].grid(axis='y', alpha=0.2)
fig.tight_layout()

## Change diagnostics and mine-level trace

Calendar joins create quarter-over-quarter and year-over-year changes without treating a missing quarter as an adjacent observation. A robust within-state score flags unusual log year-over-year changes for investigation only. Flags do not alter the state panel.

In [ ]:
diagnostics = temporal_change_diagnostics(
    development, value_end_period=DEVELOPMENT_TARGET_END
)
outlier_candidates = (
    diagnostics.loc[diagnostics['outlier_flag']].copy()
    .assign(abs_robust_z=lambda frame: frame['robust_log_yoy_z'].abs())
    .sort_values('abs_robust_z', ascending=False)
)
outlier_view = outlier_candidates.assign(
    production_million_tons=outlier_candidates['coal_production_short_tons'] / 1_000_000,
    lag4_million_tons=outlier_candidates['lag4_production'] / 1_000_000,
    yoy_change_million_tons=outlier_candidates['yoy_change'] / 1_000_000,
)[
    ['state_code', 'period', 'production_million_tons', 'lag4_million_tons',
     'yoy_change_million_tons', 'robust_log_yoy_z']
].head(20).round(3)
display(outlier_view)

flags_by_year = outlier_candidates.assign(
    year=outlier_candidates['quarter_start_date'].dt.year
).groupby('year').size()
flags_by_state = outlier_candidates.groupby('state_code').size().sort_values(ascending=False).head(12)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(flags_by_year.index, flags_by_year.values, color='#2563eb')
axes[0].set(title='Flagged changes by year', xlabel='Year', ylabel='State-quarter flags')
axes[0].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[0].grid(axis='y', alpha=0.2)
axes[1].barh(flags_by_state.index[::-1], flags_by_state.values[::-1], color='#d97706')
axes[1].set(title='States with the most flagged changes', xlabel='Flags', ylabel='')
axes[1].grid(axis='x', alpha=0.2)
fig.tight_layout()

In [ ]:
flagged_keys = outlier_candidates[['state_code', 'quarter_start_date']].drop_duplicates()
if flagged_keys.empty:
    print('No robust change flags were found.')
else:
    with duckdb.connect() as connection:
        mine_development = connection.execute(
            '''
            SELECT
                mine_id, state_code, period, quarter_start_date,
                coal_production_short_tons, production_all_null,
                production_partially_null
            FROM read_parquet(?)
            WHERE quarter_start_date BETWEEN CAST(? AS DATE) AND CAST(? AS DATE)
            ORDER BY state_code, quarter_start_date, mine_id
            ''',
            [
                str(paths['mine_quarter']),
                period_start(evaluation['modeling_start']).date(),
                DEVELOPMENT_CUTOFF.date(),
            ],
        ).fetchdf()
    flagged_mines = mine_development.merge(
        flagged_keys, on=['state_code', 'quarter_start_date'],
        how='inner', validate='many_to_one'
    )
    drilldown_summary = flagged_mines.groupby(
        ['state_code', 'period'], as_index=False
    ).agg(
        mine_rows=('mine_id', 'nunique'),
        reported_mines=('coal_production_short_tons', 'count'),
        all_null_mines=('production_all_null', 'sum'),
        partially_null_mines=('production_partially_null', 'sum'),
        reported_tons=('coal_production_short_tons', lambda values: values.sum(min_count=1)),
    )
    reported_mines = flagged_mines.dropna(subset=['coal_production_short_tons']).copy()
    reported_mines['state_period_total'] = reported_mines.groupby(
        ['state_code', 'period']
    )['coal_production_short_tons'].transform('sum')
    reported_mines['production_share'] = reported_mines['coal_production_short_tons'].div(
        reported_mines['state_period_total'].replace(0, np.nan)
    )
    top_contributors = (
        reported_mines.sort_values(
            ['state_code', 'period', 'coal_production_short_tons'],
            ascending=[True, True, False],
        )
        .groupby(['state_code', 'period'], group_keys=False).head(3)
    )
    display(drilldown_summary.head(30))
    display(
        top_contributors[
            ['state_code', 'period', 'mine_id', 'coal_production_short_tons',
             'production_share', 'production_partially_null']
        ].head(30).round({'production_share': 3})
    )

## Dynamic eligibility

Eligibility uses only observed-target availability at each origin: at least 20 observations since `2003Q1` and four consecutive observed quarters ending at the origin. Later magnitudes remain unavailable.

In [ ]:
origin_labels = [
    period_label(date)
    for date in quarter_starts(
        evaluation['validation_origins']['start'],
        evaluation['latest_forecast_origin'],
    )
]
eligibility = origin_eligibility(
    guarded_panel,
    origin_labels,
    modeling_start=evaluation['modeling_start'],
    minimum_history=evaluation['eligibility']['minimum_non_null_history'],
    recent_quarters=evaluation['eligibility']['required_consecutive_recent_observations'],
)
eligibility_counts = eligibility.groupby(
    ['origin_date', 'origin'], as_index=False
)['eligible'].sum()
development_counts = eligibility_counts.loc[
    eligibility_counts['origin_date'].between(
        period_start(evaluation['validation_origins']['start']),
        period_start(evaluation['validation_origins']['end']),
    )
]
assert len(development_counts) == evaluation['validation_origins']['count']
assert int(development_counts.iloc[0]['eligible']) == 26
assert (int(development_counts['eligible'].min()), int(development_counts['eligible'].max())) == (24, 26)

transitions = eligibility_transitions(eligibility)
display(transitions)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(eligibility_counts['origin_date'], eligibility_counts['eligible'], color='#111827', marker='o', markersize=3)
for start, end, color, label in [
    (evaluation['validation_origins']['start'], evaluation['validation_origins']['end'], '#bfdbfe', 'development'),
    (evaluation['embargo_origins']['start'], evaluation['embargo_origins']['end'], '#fde68a', 'embargo'),
    (evaluation['holdout_origins']['start'], evaluation['holdout_origins']['end'], '#fecaca', 'holdout'),
]:
    ax.axvspan(period_start(start), period_start(end) + pd.DateOffset(months=3), color=color, alpha=0.55, label=label)
ax.set(title='Eligible states by forecast origin', xlabel='', ylabel='Eligible states', ylim=(0, 28))
ax.legend(frameon=False, ncol=3)
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()

In [ ]:
eda_readiness = pd.Series(
    {
        'validated checkpoint run': manifest['run_id'],
        'development target cutoff': DEVELOPMENT_TARGET_END,
        'post-cutoff magnitudes available to EDA': False,
        'structural gaps on complete grid': int((~coverage['row_present']).sum()),
        'reported zero state-quarters in development': int(development['coal_production_short_tons'].eq(0).sum()),
        'robust change flags retained for review': len(outlier_candidates),
        'development eligible-state range': f"{int(development_counts['eligible'].min())}-{int(development_counts['eligible'].max())}",
        'rows imputed or removed': 0,
        'baseline or model evaluated': False,
    },
    name='value',
)
display(eda_readiness.to_frame())

## Decision boundary

The outputs above are diagnostic evidence, not transformation decisions. Review whether structural gaps, zeros, scale concentration, seasonal stability, and flagged mine-level composition require any predeclared treatment. Only after that review should notebook 03 reproduce the two frozen baselines under the chronological evaluation contract.